# Models

LangChainにおけるモデルは、LLMモデルのことを指します。
このノートブックは[こちらのドキュメント](https://docs.langchain.com/oss/python/langchain/models#google-gemini)をベースにしています。

## モデルの作成方法

モデルの作成方法には、Agentのノートブックで紹介した動的指定による方法以外にも、スタンドアロンでモデル単体を生成することができます。

1. エージェント作成時に動的指定：`agent = create_agent(model="モデル名",...)`のようにエージェント作成時にモデル名を
2. スタンドアロン：エージェントとは別に単体で作成する。`langchain.chat_models.init_chat_model`関数等で作成可

スタンドアロンのモデルは、いわば「ハーネスのないエージェント」のような役割を果たすため、エージェントと似た動作を実現できます。本ノートブックではこのような動作を解説していきます。

### スタンドアロンモデルの作成

スタンドアロンモデルを作成するには、`langchain.chat_models.init_chat_model`関数が簡単です。`invoke`メソッドでモデルにメッセージを送って回答を返してもらうことができます（LLMを用いたチャットボット動作）。

In [ ]:
from langchain.chat_models import init_chat_model

model = init_chat_model("google_genai:gemini-2.5-flash-lite")

response = model.invoke("Why do parrots talk?")
print(response)

以下のようにスタンドアロンモデルからエージェントを作成することもできます

In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    model=model,
    system_prompt="You are a helpful assistant who prefers concise answers",
)
result = agent.invoke(
    {"messages": [{"role": "user", "content": "Why do parrots talk?"}]}
)
print(result)

### パラメータ

モデルに各種パラメータを指定できます。

In [ ]:
model = init_chat_model(
    "google_genai:gemini-2.5-flash-lite",
    # Kwargs passed to the model:
    temperature=0.7,
    timeout=30,
    max_tokens=1000,
    max_retries=6,  # Default; increase for unreliable networks
)

response = model.invoke("Why do parrots talk?")
print(response)

## 呼び出し

Agentと同様、返答をまとめて受信する`invoke`と、逐次的に受信する`stream`が存在します。また複数メッセージを並列処理する`batch`も存在します。

### invoke

In [ ]:
response = model.invoke("Why do parrots have colorful feathers?")
print(response)

以下のようにシステムプロンプトや会話履歴を渡すと、Agentにthreadを設定した場合のような動作も実現できます

In [ ]:
conversation = [
    {"role": "system", "content": "You are a helpful assistant that translates English to French."},
    {"role": "user", "content": "Translate: I love programming."},
    {"role": "assistant", "content": "J'adore la programmation."},
    {"role": "user", "content": "Translate: I love building applications."}
]

response = model.invoke(conversation)
print(response)  # AIMessage("J'adore créer des applications.")

In [ ]:
from langchain.messages import HumanMessage, AIMessage, SystemMessage

conversation = [
    SystemMessage("You are a helpful assistant that translates English to French."),
    HumanMessage("Translate: I love programming."),
    AIMessage("J'adore la programmation."),
    HumanMessage("Translate: I love building applications.")
]

response = model.invoke(conversation)
print(response)  # AIMessage("J'adore créer des applications.")

### stream

In [ ]:
for chunk in model.stream("Why do parrots have colorful feathers?"):
    print(chunk.text, end="|", flush=True)

一度のストリームで配信されるチャンクをまとめると、`invoke`で返されるようなフルメッセージが再構成できます。

In [ ]:
full = None  # None | AIMessageChunk
for chunk in model.stream("What color is the sky?"):
    full = chunk if full is None else full + chunk
    print(full.text)
print(full.content_blocks)

### batch

複数メッセージを同時処理する`batch`を使用すると、処理の高速化やトークン削減に寄与します。

In [ ]:
responses = model.batch([
    "Why do parrots have colorful feathers?",
    "How do airplanes fly?",
    "What is quantum computing?"
])
for response in responses:
    print(response)

## Tools

Agentで`create_agent`関数に`tools`引数を指定した場合と同様に、スタンドアロンモデルでも`bind_tools`メソッドを用いることでツール呼び出しができます。

In [ ]:
from langchain.tools import tool

@tool
def get_weather(location: str) -> str:
    """Get the weather at a location."""
    return f"It's sunny in {location}."


model_with_tools = model.bind_tools([get_weather])

response = model_with_tools.invoke("What's the weather like in Boston?")
print(response)
for tool_call in response.tool_calls:
    # View tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

内部的には以下のフローでToolを呼び出しています（Toolを呼び出すかどうかの判断もモデル=LLMが実施）。基本的にはエージェントも同様の流れでToolを呼び出しています。
なお、**モデルはツールを実行するだけで、実行結果を元にした回答を生成することはしない**ことにご注意ください。

```mermaid
sequenceDiagram
    participant User
    participant Model
    participant Tools

    User->>Model: "What's the weather in SF and NYC?"
    Model->>Model: "Analyze request & decide tools needed"

    par Parallel Tool Calls
        Model->>Tools: get_weather("San Francisco")
    and
        Model->>Tools: get_weather("New York")
    end

    par Tool Execution
        Tools-->>Model: SF weather data
    and
        Tools-->>Model: NYC weather data
    end

    Note over Model: Store tool result
```

実行結果を元にした回答を生成する例を以下に示します（Agentではこの処理も自動化されている）。

In [ ]:
# Bind (potentially multiple) tools to the model
model_with_tools = model.bind_tools([get_weather])

# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "What's the weather in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)

この処理は以下のようなフローを実行していることに相当します。

```mermaid
sequenceDiagram
    participant User
    participant Model
    participant Tools

    User->>Model: "What's the weather in SF and NYC?"
    Model->>Model: "Analyze request & decide tools needed"

    par Parallel Tool Calls
        Model->>Tools: get_weather("San Francisco")
    and
        Model->>Tools: get_weather("New York")
    end

    par Tool Execution
        Tools-->>Model: SF weather data
    and
        Tools-->>Model: NYC weather data
    end

    Model->>Model: "Process results & generate response"

    Model-->>User: "SF: 72°F sunny, NYC: 68°F cloudy"
```

## Structured output

Agentで`create_agent`関数に`response_format`引数を指定した場合と同様に、スタンドアロンモデルでも`with_structured_output`メソッドを用いることで指定フォーマットに従った返答が返るようにできます。

In [ ]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    """A movie with details."""
    title: str = Field(description="The title of the movie")
    year: int = Field(description="The year the movie was released")
    director: str = Field(description="The director of the movie")
    rating: float = Field(description="The movie's rating out of 10")

model_with_structure = model.with_structured_output(Movie)
response = model_with_structure.invoke("Provide details about the movie Inception")
print(response)  # Movie(title="Inception", year=2010, director="Christopher Nolan", rating=8.8)